# 04. Queue Simulation and Design Problems

Queues model **processes in order**: elimination games, lines, rolling windows of events, rate limiters.

**Learning order:** JOSEPHUS → LUNCH QUEUE → TICKETS → REVEAL CARDS → MOVING AVERAGE → HIT COUNTER → INTERVIEW THINKING

## 1. Josephus problem (circle elimination)

`n` people in a circle, every `k`-th is removed. Who survives? Simulate with a deque: **rotate** `k-1` people from the front to the back, then remove the front.

In [1]:
from collections import deque

def josephus(n, k):
    q = deque(range(1, n + 1))
    while len(q) > 1:
        q.rotate(-(k - 1))
        q.popleft()
    return q[0]

print(josephus(5, 2))   # 3

3


## 2. Number of Students Unable to Eat Lunch

Students in a queue prefer sandwich type 0 or 1. Sandwiches are a stack. If the front student likes the top sandwich, both leave. Otherwise the student goes to the **back**. Stop when no one in the queue wants the top sandwich (a full rotation with no match).

In [2]:
def count_students(students, sandwiches):
    q, i, rotations = deque(students), 0, 0
    while q and i < len(sandwiches) and rotations < len(q):
        if q[0] == sandwiches[i]:
            q.popleft()
            i += 1
            rotations = 0
        else:
            q.append(q.popleft())
            rotations += 1
    return len(q)

print(count_students([1, 1, 0, 0], [0, 1, 0, 1]), count_students([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))   # 0 3

0 3


## 3. Time Needed to Buy Tickets

Each person buys one ticket then goes to the back if they still need more. How long until person `k` finishes? Simulate with a queue of `(index, remaining)`.

(Counting solution: person `i` before `k` buys `min(t[i], t[k])`, after `k` buys `min(t[i], t[k] - 1)`. O(n).)

In [3]:
def time_to_buy(tickets, k):
    q, time = deque((i, t) for i, t in enumerate(tickets)), 0
    while q:
        i, remaining = q.popleft()
        time += 1
        if remaining - 1 > 0:
            q.append((i, remaining - 1))
        elif i == k:
            return time

print(time_to_buy([2, 3, 2], 2), time_to_buy([5, 1, 1, 1], 0))   # 6 8

6 8


## 4. Reveal Cards in Increasing Order

Reverse the process by **simulating positions**. Sort the deck. Keep a queue of the **index positions**. For each smallest card: it goes to the first position in the queue; then the next position moves to the back (the "skip" step).

In [4]:
def deck_revealed_increasing(deck):
    n = len(deck)
    res = [0] * n
    q = deque(range(n))
    for card in sorted(deck):
        res[q.popleft()] = card
        if q:
            q.append(q.popleft())
    return res

print(deck_revealed_increasing([17, 13, 11, 2, 3, 5, 7]))   # [2, 13, 3, 11, 5, 17, 7]

[2, 13, 3, 11, 5, 17, 7]


## 5. Moving Average from a Data Stream

Keep the last `size` values in a deque and a **running sum**. Add the new value, subtract the removed one. O(1) per call.

In [5]:
class MovingAverage:
    def __init__(self, size):
        self.size, self.q, self.total = size, deque(), 0

    def next(self, val):
        self.q.append(val)
        self.total += val
        if len(self.q) > self.size:
            self.total -= self.q.popleft()
        return self.total / len(self.q)

m = MovingAverage(3)
print([round(m.next(v), 2) for v in [1, 10, 3, 5]])   # [1.0, 5.5, 4.67, 6.0]

[1.0, 5.5, 4.67, 6.0]


## 6. Hit Counter (rate limiter idea)

Count hits in the last 300 seconds. Store timestamps in a deque, drop old ones on every query. This is the **sliding window log** rate limiter used in real systems (APIs allowing N requests per time window).

In [6]:
class HitCounter:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        self.q.append(timestamp)

    def get_hits(self, timestamp):
        while self.q and self.q[0] <= timestamp - 300:
            self.q.popleft()
        return len(self.q)

h = HitCounter()
h.hit(1); h.hit(2); h.hit(3)
print(h.get_hits(4))     # 3
h.hit(300)
print(h.get_hits(300))   # 4
print(h.get_hits(301))   # 3

3
4
3


## 7. Interview thinking

**Signals:** "simulate a process in rounds", "circle elimination", "last N seconds/items", "design a counter/rate limiter".

**Say:** "I simulate with a deque. For windows over time, I append new events and pop expired ones from the front, so each event is handled twice at most."

**Common mistakes:** infinite loop in simulations without a stop condition (see the lunch queue's `rotations`), off-by-one on window boundaries (`<` vs `<=`), recomputing sums instead of keeping a running total.

## Quick Revision

- Deque `rotate(-(k-1))` + `popleft` = Josephus.
- Simulation needs a stop condition.
- Time-window problems: append new, pop expired, keep a running sum/count.
- Rate limiter = sliding window log.